# 04 — RSNA Knee Baseline V1 Train

Frozen **DINOv2-Small** image features + a compact label-specific attention head for the 12-target RSNA Knee Abnormality Detection task.

Baseline contract: five masked MRI slots, geometry-sorted DICOM, 140 mm physical crop, 256 px adjacent-slice 2.5D, eight centers per available slot, soft L1 family-mean supervision, 5-fold `GroupKFold(report_group)`, weak-validation-only early stopping, and an independent 58-study gold audit.

This notebook is self-contained for a fresh Kaggle GPU session with internet disabled. Required competition data, notebook 02/03 artifacts, and a local Hugging Face DINOv2-Small model must be attached as inputs.


In [ ]:
# 1. Offline environment, imports, and locked configuration
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter
import copy, hashlib, json, math, random, re, traceback, warnings, zipfile

import numpy as np
import pandas as pd
import pydicom
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

warnings.filterwarnings("ignore")

SEED = 42
LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
SLOTS = ["SAG_FLUID", "SAG_NONFLUID", "COR_FLUID", "AX_FLUID", "COR_NONFLUID"]
ALL_ROUTING_SLOTS = {
    "SAG_FLUID": ("Sagittal", 1), "SAG_NONFLUID": ("Sagittal", 0),
    "COR_FLUID": ("Coronal", 1), "COR_NONFLUID": ("Coronal", 0),
    "AX_FLUID": ("Axial", 1), "AX_NONFLUID": ("Axial", 0),
}

CFG = {
    "seed": SEED, "n_folds": 5, "crop_mm": 140.0, "image_size": 256,
    "centers_per_slot": 8, "percentiles": [1.0, 99.0],
    "feature_batch_size": 32, "feature_workers": 0,
    "head_dim": 256, "head_batch_size": 128, "max_epochs": 20,
    "patience": 4, "learning_rate": 3e-4, "weight_decay": 1e-4,
    "targets": LABELS, "slots": SLOTS, "encoder": "DINOv2-Small",
    "encoder_frozen": True, "loss": "BCEWithLogitsLoss_soft_targets",
    "weak_auc_effective_positive_threshold": 0.5,
}

OUT = Path("/kaggle/working/rsna_knee_baseline_v1")
OUT.mkdir(parents=True, exist_ok=True)
RUN_START = perf_counter()

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("torch:", torch.__version__, "device:", DEVICE, "GPUs:", GPU_NAMES)


## 2. Discover and validate all attached inputs

Discovery is schema-aware. DINO discovery is deliberately strict: it accepts only a local directory whose path contains `dinov2`, whose `config.json` declares DINOv2, and whose hidden size is 384 (DINOv2-Small). There is no random-weight or alternate-backbone fallback.


In [ ]:
def read_cols(path):
    try: return set(pd.read_csv(path, nrows=2).columns)
    except Exception: return set()

def find_competition_root():
    valid = []
    for p in Path("/kaggle/input").rglob("train.csv"):
        root = p.parent
        required_paths = ["train_series.csv", "train_series", "test.csv", "test_series.csv", "test_series"]
        if not all((root / x).exists() for x in required_paths): continue
        if {"StudyInstanceUID", "Report", "ACL", "MCL"}.issubset(read_cols(p)):
            valid.append(root)
    if not valid:
        raise FileNotFoundError("RSNA Knee competition root not found with all train/test files and DICOM directories.")
    preferred = [x for x in valid if "/competitions/" in str(x).replace("\\", "/")]
    return preferred[0] if preferred else sorted(valid, key=str)[0]

def find_exact_file(name, required_cols=None):
    hits = []
    for p in Path("/kaggle/input").rglob(name):
        if required_cols is None or set(required_cols).issubset(read_cols(p)): hits.append(p)
    if not hits: raise FileNotFoundError(f"Required attached artifact not found: {name}")
    hits = sorted(hits, key=lambda p: (len(str(p)), str(p)))
    print(f"{name} candidates:", [str(x) for x in hits])
    return hits[0]

def find_dinov2_small():
    all_configs = sorted(Path("/kaggle/input").rglob("config.json"), key=str)
    named = [p.parent for p in all_configs if "dinov2" in str(p.parent).lower()]
    compatible = []
    for d in named:
        try:
            cfg = json.loads((d / "config.json").read_text(encoding="utf-8"))
            model_type = str(cfg.get("model_type", "")).lower()
            hidden = int(cfg.get("hidden_size", -1))
            has_weights = any((d / n).exists() for n in [
                "model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"
            ])
            if "dinov2" in model_type and hidden == 384 and has_weights: compatible.append(d)
        except Exception:
            pass
    if not compatible:
        print("No compatible local DINOv2-Small model was found.")
        print("Candidate config directories containing 'dinov2':")
        for x in named: print(" -", x)
        print("Other config.json directories (first 50):")
        for x in all_configs[:50]: print(" -", x.parent)
        raise FileNotFoundError(
            "Attach an offline Hugging Face DINOv2-Small model directory containing config.json "
            "(model_type=dinov2, hidden_size=384) and pretrained weight files."
        )
    return compatible[0]

COMP_ROOT = find_competition_root()
SILVER_PATH = find_exact_file("L1_family_mean_RECOMMENDED.csv", ["StudyInstanceUID"] + LABELS)
TRAIN_MANIFEST_PATH = find_exact_file("baseline_series_manifest.csv", ["StudyInstanceUID", "report_group"])
DINO_PATH = find_dinov2_small()

TRAIN_DICOM_ROOT = COMP_ROOT / "train_series"
TEST_DICOM_ROOT = COMP_ROOT / "test_series"
print("Competition:", COMP_ROOT)
print("Silver:", SILVER_PATH)
print("Train manifest:", TRAIN_MANIFEST_PATH)
print("DINOv2-Small:", DINO_PATH)


In [ ]:
# 3. Load tables and enforce supervision-integrity assertions
train = pd.read_csv(COMP_ROOT / "train.csv", dtype={"StudyInstanceUID": str})
test = pd.read_csv(COMP_ROOT / "test.csv", dtype={"StudyInstanceUID": str})
test_series = pd.read_csv(COMP_ROOT / "test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
silver = pd.read_csv(SILVER_PATH, dtype={"StudyInstanceUID": str})
train_manifest = pd.read_csv(TRAIN_MANIFEST_PATH, dtype={"StudyInstanceUID": str})

assert all(c in train.columns for c in LABELS)
assert all(c in silver.columns for c in LABELS)
assert all(f"{s}__SeriesInstanceUID" in train_manifest.columns for s in SLOTS)
gold = train.loc[train[LABELS].notna().all(axis=1), ["StudyInstanceUID"] + LABELS].copy()
gold_ids = set(gold.StudyInstanceUID)
silver_ids = set(silver.StudyInstanceUID)

assert len(gold) == 58, f"Expected 58 gold studies, got {len(gold)}"
assert len(silver) == 4349, f"Expected 4349 weak studies, got {len(silver)}"
assert gold_ids.isdisjoint(silver_ids)
assert silver["StudyInstanceUID"].is_unique
assert not silver[LABELS].isna().any().any()
assert np.isfinite(silver[LABELS].to_numpy(float)).all()
assert set(train.StudyInstanceUID) == set(train_manifest.StudyInstanceUID)

weak_manifest = train_manifest[train_manifest.StudyInstanceUID.isin(silver_ids)].copy()
weak_manifest = weak_manifest.merge(silver, on="StudyInstanceUID", how="left", validate="one_to_one")
gold_manifest = train_manifest[train_manifest.StudyInstanceUID.isin(gold_ids)].copy()
gold_manifest = gold[["StudyInstanceUID"]].merge(gold_manifest, on="StudyInstanceUID", how="left", validate="one_to_one")
assert len(weak_manifest) == 4349 and len(gold_manifest) == 58

print("weak:", len(weak_manifest), "gold:", len(gold_manifest), "test:", len(test))


## 4. Build the test series manifest with notebook 03's exact rule

Train series are never reselected. Test candidates use the same slot mapping and ranking precedence from notebook 03: usable slice count, 140 mm FOV, smaller pixel spacing, distance from 32 slices, then `SeriesInstanceUID`.


In [ ]:
def dcm_paths(series_dir):
    try:
        return [Path(e.path) for e in os.scandir(series_dir) if e.is_file() and e.name.lower().endswith(".dcm")]
    except Exception:
        return []

def as_float(x):
    try: return float(x)
    except Exception: return np.nan

def test_series_header(row):
    study, series_uid = str(row.StudyInstanceUID), str(row.SeriesInstanceUID)
    paths = dcm_paths(TEST_DICOM_ROOT / study / series_uid)
    out = {"n_slices": len(paths), "Rows": np.nan, "Columns": np.nan,
           "PixelSpacingRow": np.nan, "PixelSpacingCol": np.nan,
           "PixelSpacingMean": np.nan, "FOVRow_mm": np.nan, "FOVCol_mm": np.nan,
           "SliceThickness": np.nan, "Manufacturer": "", "SeriesDescription": ""}
    if not paths: return out
    try:
        ds = pydicom.dcmread(str(paths[0]), stop_before_pixels=True, force=True)
        ps = getattr(ds, "PixelSpacing", None)
        psr = as_float(ps[0]) if ps is not None and len(ps) >= 2 else np.nan
        psc = as_float(ps[1]) if ps is not None and len(ps) >= 2 else np.nan
        rr, cc = as_float(getattr(ds, "Rows", np.nan)), as_float(getattr(ds, "Columns", np.nan))
        out.update({
            "Rows": rr, "Columns": cc, "PixelSpacingRow": psr, "PixelSpacingCol": psc,
            "PixelSpacingMean": np.nanmean([psr, psc]),
            "FOVRow_mm": rr * psr if np.isfinite(rr * psr) else np.nan,
            "FOVCol_mm": cc * psc if np.isfinite(cc * psc) else np.nan,
            "SliceThickness": as_float(getattr(ds, "SliceThickness", np.nan)),
            "Manufacturer": str(getattr(ds, "Manufacturer", "")),
            "SeriesDescription": str(getattr(ds, "SeriesDescription", "")),
        })
    except Exception:
        pass
    return out

def route_slot(row):
    try: fluid = int(row["Fluid_Sensitive"])
    except Exception: return None
    plane = str(row["Anatomical_Plane"])
    for slot, (p, f) in ALL_ROUTING_SLOTS.items():
        if plane == p and fluid == f: return slot
    return None

print("Reading one test header per series...")
with ThreadPoolExecutor(max_workers=8) as ex:
    hdr = list(ex.map(test_series_header, test_series.itertuples(index=False)))
test_series = pd.concat([test_series.reset_index(drop=True), pd.DataFrame(hdr)], axis=1)
test_series["slot"] = test_series.apply(route_slot, axis=1)
test_series["usable_slice_count"] = test_series["n_slices"].between(12, 100).astype(int)
test_series["fov_ok"] = (test_series[["FOVRow_mm", "FOVCol_mm"]].min(axis=1) >= CFG["crop_mm"]).astype(int)
test_series["spacing_rank_value"] = test_series["PixelSpacingMean"].fillna(999.0)
test_series["slice_count_distance"] = (test_series["n_slices"] - 32).abs()

ranked_test = test_series.sort_values(
    ["StudyInstanceUID", "slot", "usable_slice_count", "fov_ok", "spacing_rank_value", "slice_count_distance", "SeriesInstanceUID"],
    ascending=[True, True, False, False, True, True, True]
)
selected_test = ranked_test.dropna(subset=["slot"]).groupby(["StudyInstanceUID", "slot"], as_index=False).first()
test_manifest = test[["StudyInstanceUID"]].copy()
for slot in SLOTS:
    sub = selected_test[selected_test.slot.eq(slot)][["StudyInstanceUID", "SeriesInstanceUID"]].rename(
        columns={"SeriesInstanceUID": f"{slot}__SeriesInstanceUID"})
    test_manifest = test_manifest.merge(sub, on="StudyInstanceUID", how="left", validate="one_to_one")
assert list(test_manifest.StudyInstanceUID) == list(test.StudyInstanceUID)
display(test_manifest.head())


## 5. Robust DICOM geometry sorting and physical preprocessing

Unreadable headers/slices are logged and skipped. The primary order is the projection of `ImagePositionPatient` onto the slice normal derived from `ImageOrientationPatient`. Only a complete `InstanceNumber` order is used as fallback. Filename/SOP UID order is never used as a primary ordering method.


In [ ]:
FAIL = {
    "geometry_fallback_series": 0, "dicom_header_failures": 0,
    "dicom_decode_failures": 0, "series_failures": 0, "feature_failures": 0,
    "feature_tokens_extracted": 0,
}

def read_sort_record(path):
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        iop, ipp = getattr(ds, "ImageOrientationPatient", None), getattr(ds, "ImagePositionPatient", None)
        position = np.nan
        if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
            iop = np.asarray(iop[:6], dtype=np.float64); ipp = np.asarray(ipp[:3], dtype=np.float64)
            normal = np.cross(iop[:3], iop[3:])
            if np.isfinite(iop).all() and np.isfinite(ipp).all() and np.linalg.norm(normal) > 1e-8:
                position = float(np.dot(ipp, normal))
        instance = as_float(getattr(ds, "InstanceNumber", np.nan))
        return {"path": path, "position": position, "instance": instance}
    except Exception:
        FAIL["dicom_header_failures"] += 1
        return None

def geometry_sorted_paths(series_dir):
    records = [read_sort_record(p) for p in dcm_paths(series_dir)]
    records = [r for r in records if r is not None]
    if len(records) < 3: raise RuntimeError("fewer than 3 readable DICOM headers")
    if all(np.isfinite(r["position"]) for r in records):
        records.sort(key=lambda r: r["position"])
    elif all(np.isfinite(r["instance"]) for r in records):
        FAIL["geometry_fallback_series"] += 1
        records.sort(key=lambda r: r["instance"])
    else:
        raise RuntimeError("incomplete geometry and incomplete InstanceNumber; refusing filename order")
    return [r["path"] for r in records]

def robust_01(arr):
    arr = np.asarray(arr, dtype=np.float32)
    finite = np.isfinite(arr)
    if not finite.any(): return np.zeros_like(arr, dtype=np.float32)
    fill = float(np.median(arr[finite])); arr = np.where(finite, arr, fill)
    lo, hi = np.percentile(arr, CFG["percentiles"])
    if not np.isfinite(lo + hi) or hi <= lo + 1e-6: return np.zeros_like(arr, dtype=np.float32)
    return np.clip((np.clip(arr, lo, hi) - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

def decode_preprocess(path):
    try:
        ds = pydicom.dcmread(str(path), force=True)
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * as_float(getattr(ds, "RescaleSlope", 1.0)) + as_float(getattr(ds, "RescaleIntercept", 0.0))
        finite = np.isfinite(arr)
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1" and finite.any():
            arr = float(np.nanmax(arr[finite]) + np.nanmin(arr[finite])) - arr
        arr = robust_01(arr)
        ps = getattr(ds, "PixelSpacing", None)
        if ps is None or len(ps) < 2: raise ValueError("missing PixelSpacing")
        row_sp, col_sp = float(ps[0]), float(ps[1])
        if not (np.isfinite(row_sp) and np.isfinite(col_sp) and row_sp > 0 and col_sp > 0):
            raise ValueError("invalid PixelSpacing")
        h, w = arr.shape[-2:]
        crop_h = min(h, max(1, int(round(CFG["crop_mm"] / row_sp))))
        crop_w = min(w, max(1, int(round(CFG["crop_mm"] / col_sp))))
        y0, x0 = max(0, (h - crop_h) // 2), max(0, (w - crop_w) // 2)
        crop = arr[y0:y0 + crop_h, x0:x0 + crop_w]
        t = torch.from_numpy(crop)[None, None]
        t = F.interpolate(t, size=(CFG["image_size"], CFG["image_size"]), mode="bilinear", align_corners=False)
        return t[0, 0].numpy().astype(np.float32)
    except Exception:
        FAIL["dicom_decode_failures"] += 1
        return None

IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], np.float32)[:, None, None]
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], np.float32)[:, None, None]

def load_series_triplets(series_dir):
    try:
        paths = geometry_sorted_paths(series_dir)
        n = len(paths)
        centers = np.rint(np.linspace(1, n - 2, CFG["centers_per_slot"])).astype(int)
        decoded = {}
        images, positions = [], []
        for c in centers:
            inds = [int(c - 1), int(c), int(c + 1)]
            for j in inds:
                if j not in decoded: decoded[j] = decode_preprocess(paths[j])
            if any(decoded[j] is None for j in inds):
                FAIL["feature_failures"] += 1
                continue
            x = np.stack([decoded[j] for j in inds], axis=0)
            x = (x - IMAGENET_MEAN) / IMAGENET_STD
            images.append(x.astype(np.float32)); positions.append(float(c / (n - 1)))
        return images, positions
    except Exception:
        FAIL["series_failures"] += 1
        return [], []


## 6. Load the frozen offline DINOv2-Small encoder


In [ ]:
encoder = AutoModel.from_pretrained(str(DINO_PATH), local_files_only=True)
model_type = str(getattr(encoder.config, "model_type", "")).lower()
hidden_size = int(getattr(encoder.config, "hidden_size", -1))
assert "dinov2" in model_type and hidden_size == 384, (
    f"Expected DINOv2-Small (hidden_size=384), got model_type={model_type}, hidden_size={hidden_size}"
)
encoder.eval().to(DEVICE)
for p in encoder.parameters(): p.requires_grad_(False)
CFG["dino_path"] = str(DINO_PATH)
CFG["dino_model_type"] = model_type
CFG["dino_hidden_size"] = hidden_size
print(encoder.config)


## 7. Reusable frozen-feature cache

Each split cache contains fixed indexing for at most 40 positions per study (`5 slots × 8 centers`), float16 CLS embeddings, slot IDs, normalized slice positions, and a boolean validity mask. Invalid/missing positions are always excluded by the head's masked attention.


In [ ]:
@torch.inference_mode()
def encode_batch(images):
    x = torch.from_numpy(np.stack(images)).to(DEVICE, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        out = encoder(pixel_values=x)
        z = out.last_hidden_state[:, 0]
    return z.float().cpu().numpy()

def extract_cache(name, manifest, dicom_root):
    start = perf_counter()
    n, T, D = len(manifest), len(SLOTS) * CFG["centers_per_slot"], hidden_size
    features = np.zeros((n, T, D), dtype=np.float16)
    positions = np.zeros((n, T), dtype=np.float16)
    slot_ids = np.tile(np.repeat(np.arange(len(SLOTS)), CFG["centers_per_slot"]), (n, 1)).astype(np.int8)
    mask = np.zeros((n, T), dtype=bool)
    pending_images, pending_keys = [], []

    def flush():
        nonlocal pending_images, pending_keys
        if not pending_images: return
        z = encode_batch(pending_images)
        for emb, (i, token_idx, pos) in zip(z, pending_keys):
            features[i, token_idx] = emb.astype(np.float16)
            positions[i, token_idx] = np.float16(pos)
            mask[i, token_idx] = True
            FAIL["feature_tokens_extracted"] += 1
        pending_images, pending_keys = [], []

    for i, row in enumerate(manifest.itertuples(index=False)):
        study = str(row.StudyInstanceUID)
        row_dict = row._asdict()
        # itertuples sanitizes names containing double underscores; use iloc for exact columns.
        series_row = manifest.iloc[i]
        for slot_idx, slot in enumerate(SLOTS):
            series_uid = series_row.get(f"{slot}__SeriesInstanceUID", np.nan)
            if pd.isna(series_uid) or str(series_uid).strip() == "": continue
            images, pos = load_series_triplets(dicom_root / study / str(series_uid))
            for j, (img, p) in enumerate(zip(images[:CFG["centers_per_slot"]], pos[:CFG["centers_per_slot"]])):
                pending_images.append(img)
                pending_keys.append((i, slot_idx * CFG["centers_per_slot"] + j, p))
                if len(pending_images) >= CFG["feature_batch_size"]: flush()
        if (i + 1) % 250 == 0: print(f"{name}: {i+1}/{n}")
    flush()

    cache_path = OUT / f"{name}_features.npz"
    np.savez_compressed(cache_path, features=features, positions=positions, slot_ids=slot_ids,
                        mask=mask, study_ids=manifest.StudyInstanceUID.astype(str).to_numpy())
    rows = []
    for i, uid in enumerate(manifest.StudyInstanceUID.astype(str)):
        for t in range(T):
            rows.append({"StudyInstanceUID": uid, "cache_row": i, "token_index": t,
                         "slot_id": int(slot_ids[i, t]), "slot": SLOTS[int(slot_ids[i, t])],
                         "normalized_slice_position": float(positions[i, t]), "valid_mask": bool(mask[i, t])})
    pd.DataFrame(rows).to_csv(OUT / f"{name}_feature_manifest.csv", index=False)
    runtime = perf_counter() - start
    print(name, "cache:", cache_path, "valid tokens:", int(mask.sum()), "seconds:", round(runtime, 1))
    return {"features": features, "positions": positions, "slot_ids": slot_ids,
            "mask": mask, "study_ids": manifest.StudyInstanceUID.astype(str).to_numpy(),
            "runtime_seconds": runtime, "path": str(cache_path)}

cache_stats_before = copy.deepcopy(FAIL)
weak_cache = extract_cache("train", weak_manifest, TRAIN_DICOM_ROOT)
gold_cache = extract_cache("gold", gold_manifest, TRAIN_DICOM_ROOT)
test_cache = extract_cache("test", test_manifest, TEST_DICOM_ROOT)
FEATURE_RUNTIME = sum(x["runtime_seconds"] for x in [weak_cache, gold_cache, test_cache])
print("Feature failure counters:", FAIL)


## 8. Label-specific masked-attention study head

The head adds learned slot identity and an MLP encoding of normalized anatomical slice position to each projected DINO token. Twelve learned pathology queries independently pool the same valid token set; twelve independent linear readouts produce logits. A fully missing study yields a zero pooled vector and finite bias-only predictions rather than a fabricated image token.


In [ ]:
class StudyDataset(Dataset):
    def __init__(self, cache, targets=None):
        self.f = cache["features"]; self.p = cache["positions"]
        self.s = cache["slot_ids"]; self.m = cache["mask"]
        self.y = targets
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        item = (torch.from_numpy(self.f[i].astype(np.float32)), torch.from_numpy(self.s[i].astype(np.int64)),
                torch.from_numpy(self.p[i].astype(np.float32)), torch.from_numpy(self.m[i]))
        if self.y is not None: item += (torch.from_numpy(self.y[i].astype(np.float32)),)
        return item

class LabelAttentionHead(nn.Module):
    def __init__(self, in_dim=384, dim=256, n_slots=5, n_labels=12):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, dim), nn.GELU())
        self.slot_emb = nn.Embedding(n_slots, dim)
        self.pos_mlp = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        self.token_norm = nn.LayerNorm(dim)
        self.queries = nn.Parameter(torch.randn(n_labels, dim) / math.sqrt(dim))
        self.out_weight = nn.Parameter(torch.randn(n_labels, dim) / math.sqrt(dim))
        self.out_bias = nn.Parameter(torch.zeros(n_labels))

    def forward(self, features, slot_ids, positions, mask, return_attention=False):
        x = self.proj(features) + self.slot_emb(slot_ids) + self.pos_mlp(torch.nan_to_num(positions)[..., None])
        x = self.token_norm(x)
        score = torch.einsum("ld,btd->blt", self.queries, x) / math.sqrt(x.shape[-1])
        valid = mask[:, None, :]
        score = score.masked_fill(~valid, -1e4)
        attn = torch.softmax(score, dim=-1) * valid.float()
        attn = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        pooled = torch.einsum("blt,btd->bld", attn, x)
        logits = (pooled * self.out_weight[None]).sum(-1) + self.out_bias
        return (logits, attn) if return_attention else logits


## 9. Metrics, grouped folds, training, and gold audit

Soft targets are used unchanged in `BCEWithLogitsLoss`. Standard ROC-AUC requires binary ground truth, so weak-validation AUC uses a documented effective class (`soft target >= 0.5`) solely for early stopping. A one-class target is reported as invalid and excluded only from that fold's macro AUC. Gold labels are evaluated as their original binary values and never influence checkpoint selection.


In [ ]:
def metric_rows(y, p, labels=LABELS):
    rows = []
    for j, label in enumerate(labels):
        yt, pt = np.asarray(y[:, j]), np.asarray(p[:, j])
        valid = np.isfinite(yt) & np.isfinite(pt)
        yy, pp = yt[valid], pt[valid]
        if len(np.unique(yy)) < 2:
            auc = ap = np.nan
        else:
            auc = roc_auc_score(yy, pp); ap = average_precision_score(yy, pp)
        rows.append({"label": label, "n": len(yy), "n_positive": int((yy == 1).sum()), "auc": auc, "ap": ap})
    return pd.DataFrame(rows)

@torch.inference_mode()
def predict_head(model, cache, batch_size=256, save_attention=False):
    loader = DataLoader(StudyDataset(cache), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    probs, debug = [], []
    model.eval()
    for batch_idx, (f, s, p, m) in enumerate(loader):
        f, s, p, m = [x.to(DEVICE, non_blocking=True) for x in (f, s, p, m)]
        if save_attention and batch_idx == 0:
            logits, attn = model(f, s, p, m, return_attention=True); debug.append(attn.cpu().numpy())
        else: logits = model(f, s, p, m)
        probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs), (debug[0] if debug else None)

def train_fold(fold, tr_idx, va_idx, targets):
    train_groups = set(weak_manifest.iloc[tr_idx].report_group.astype(str))
    val_groups = set(weak_manifest.iloc[va_idx].report_group.astype(str))
    train_ids = set(weak_manifest.iloc[tr_idx].StudyInstanceUID.astype(str))
    val_ids = set(weak_manifest.iloc[va_idx].StudyInstanceUID.astype(str))
    assert train_groups.isdisjoint(val_groups)
    assert train_ids.isdisjoint(gold_ids) and val_ids.isdisjoint(gold_ids)

    model = LabelAttentionHead(hidden_size, CFG["head_dim"], len(SLOTS), len(LABELS)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"], weight_decay=CFG["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()
    g = torch.Generator(); g.manual_seed(SEED + fold)
    ds = StudyDataset({k: weak_cache[k][tr_idx] for k in ["features", "positions", "slot_ids", "mask"]}, targets[tr_idx])
    loader = DataLoader(ds, batch_size=CFG["head_batch_size"], shuffle=True, generator=g,
                        num_workers=0, pin_memory=True, drop_last=False)
    val_cache = {k: weak_cache[k][va_idx] for k in ["features", "positions", "slot_ids", "mask"]}
    best_auc, best_epoch, stale, best_state = -np.inf, -1, 0, None

    for epoch in range(CFG["max_epochs"]):
        model.train(); losses = []
        for f, s, p, m, y in loader:
            f, s, p, m, y = [x.to(DEVICE, non_blocking=True) for x in (f, s, p, m, y)]
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(f, s, p, m), y)
            loss.backward(); opt.step(); losses.append(float(loss.detach().cpu()))
        val_prob, _ = predict_head(model, val_cache)
        val_effective = (targets[va_idx] >= CFG["weak_auc_effective_positive_threshold"]).astype(np.uint8)
        vm = metric_rows(val_effective, val_prob)
        macro = float(vm.auc.mean(skipna=True))
        invalid = vm.loc[vm.auc.isna(), "label"].tolist()
        print(f"fold={fold} epoch={epoch:02d} loss={np.mean(losses):.5f} weak_macro_auc={macro:.6f} invalid={invalid}")
        if np.isfinite(macro) and macro > best_auc + 1e-5:
            best_auc, best_epoch, stale = macro, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= CFG["patience"]: break
    if best_state is None: raise RuntimeError(f"Fold {fold} never produced a valid weak macro AUC")
    model.load_state_dict(best_state)
    torch.save({"state_dict": best_state, "fold": fold, "best_epoch": best_epoch,
                "weak_val_macro_auc": best_auc, "config": CFG}, OUT / f"fold_{fold}_head.pt")
    return model, best_epoch, best_auc

targets = weak_manifest[LABELS].to_numpy(np.float32)
groups = weak_manifest.report_group.astype(str).to_numpy()
fold_id = np.full(len(weak_manifest), -1, dtype=np.int8)
splits = list(GroupKFold(n_splits=CFG["n_folds"]).split(weak_manifest, groups=groups))
for f, (_, vi) in enumerate(splits): fold_id[vi] = f
assert (fold_id >= 0).all()
pd.DataFrame({"StudyInstanceUID": weak_manifest.StudyInstanceUID, "report_group": groups, "fold": fold_id}).to_csv(
    OUT / "fold_assignments.csv", index=False)

weak_metric_frames, gold_metric_frames, gold_fold_probs, fold_summary = [], [], [], []
gold_y = gold.set_index("StudyInstanceUID").loc[gold_manifest.StudyInstanceUID, LABELS].to_numpy(np.uint8)

for fold, (tr_idx, va_idx) in enumerate(splits):
    seed_everything(SEED + fold)
    model, best_epoch, best_auc = train_fold(fold, tr_idx, va_idx, targets)
    val_cache = {k: weak_cache[k][va_idx] for k in ["features", "positions", "slot_ids", "mask"]}
    val_prob, _ = predict_head(model, val_cache)
    vm = metric_rows((targets[va_idx] >= 0.5).astype(np.uint8), val_prob)
    vm = pd.concat([vm, pd.DataFrame([{"label": "MACRO", "n": len(va_idx), "n_positive": int((targets[va_idx] >= 0.5).sum()),
                                      "auc": float(vm.auc.mean(skipna=True)), "ap": float(vm.ap.mean(skipna=True))}])], ignore_index=True)
    vm.insert(0, "fold", fold); weak_metric_frames.append(vm)

    gp, attn = predict_head(model, gold_cache, save_attention=(fold == 0))
    gold_fold_probs.append(gp)
    gm = metric_rows(gold_y, gp)
    gold_macro = float(gm.auc.mean(skipna=True)); gold_macro_ap = float(gm.ap.mean(skipna=True))
    gm = pd.concat([gm, pd.DataFrame([{"label": "MACRO", "n": len(gold_y), "n_positive": int(gold_y.sum()),
                                      "auc": gold_macro, "ap": gold_macro_ap}])], ignore_index=True)
    gm.insert(0, "fold", fold); gold_metric_frames.append(gm)
    fold_summary.append({"fold": fold, "best_epoch": best_epoch,
                         "weak_val_macro_auc": best_auc, "gold_macro_auc": gold_macro,
                         "gold_macro_ap": gold_macro_ap})
    if attn is not None:
        np.savez_compressed(OUT / "debug_attention_fold0.npz", attention=attn,
                            study_ids=gold_cache["study_ids"][:len(attn)], slots=np.asarray(SLOTS))
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

weak_cv_metrics = pd.concat(weak_metric_frames, ignore_index=True)
gold_fold_metrics = pd.concat(gold_metric_frames, ignore_index=True)
weak_cv_metrics.to_csv(OUT / "weak_cv_metrics.csv", index=False)
gold_fold_metrics.to_csv(OUT / "gold_fold_metrics.csv", index=False)
display(pd.DataFrame(fold_summary))


## 10. Independent 58-study ensemble gold audit

These are small-sample audit metrics only. No fold, epoch, threshold, or model component is selected from them.


In [ ]:
gold_stack = np.stack(gold_fold_probs)
gold_ensemble = gold_stack.mean(axis=0)
gold_ensemble_metrics = metric_rows(gold_y, gold_ensemble)
macro_row = {"label": "MACRO", "n": len(gold_y), "n_positive": int(gold_y.sum()),
             "auc": float(gold_ensemble_metrics.auc.mean(skipna=True)),
             "ap": float(gold_ensemble_metrics.ap.mean(skipna=True))}
gold_ensemble_metrics = pd.concat([gold_ensemble_metrics, pd.DataFrame([macro_row])], ignore_index=True)
gold_ensemble_metrics.to_csv(OUT / "gold_ensemble_metrics.csv", index=False)

gold_predictions = pd.DataFrame({"StudyInstanceUID": gold_manifest.StudyInstanceUID})
for f in range(CFG["n_folds"]):
    for j, label in enumerate(LABELS): gold_predictions[f"fold_{f}__{label}"] = gold_stack[f, :, j]
for j, label in enumerate(LABELS):
    gold_predictions[f"ensemble__{label}"] = gold_ensemble[:, j]
    gold_predictions[f"truth__{label}"] = gold_y[:, j]
gold_predictions.to_csv(OUT / "gold_predictions.csv", index=False)
display(gold_ensemble_metrics)


## 11. Five-fold test inference and submission


In [ ]:
test_fold_probs = []
for fold in range(CFG["n_folds"]):
    ckpt = torch.load(OUT / f"fold_{fold}_head.pt", map_location="cpu")
    model = LabelAttentionHead(hidden_size, CFG["head_dim"], len(SLOTS), len(LABELS)).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    pp, _ = predict_head(model, test_cache)
    test_fold_probs.append(pp)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

predictions = np.stack(test_fold_probs).mean(axis=0)
submission = pd.DataFrame(predictions, columns=LABELS)
submission.insert(0, "StudyInstanceUID", test.StudyInstanceUID.astype(str).to_numpy())
assert list(submission.StudyInstanceUID) == list(test.StudyInstanceUID.astype(str))
assert submission.shape[1] == 13
assert np.isfinite(predictions).all()
assert predictions.min() >= 0 and predictions.max() <= 1
submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.to_csv(OUT / "submission.csv", index=False)
display(submission.head())
print("submission:", submission.shape, "range:", float(predictions.min()), float(predictions.max()))


## 12. Results report and compact artifact bundle

Feature-cache arrays stay outside the diagnostics ZIP. The ZIP contains configuration, feature manifests, fold assignments, head checkpoints, metrics, gold predictions, submission, debug attention, and the readable run report.


In [ ]:
CFG.update({
    "competition_root": str(COMP_ROOT), "silver_path": str(SILVER_PATH),
    "series_manifest_path": str(TRAIN_MANIFEST_PATH), "gpu_names": GPU_NAMES,
    "weak_count": len(weak_manifest), "gold_count": len(gold_manifest), "test_count": len(test_manifest),
})
with open(OUT / "config.json", "w", encoding="utf-8") as f: json.dump(CFG, f, indent=2)

slot_lines = []
for split_name, mf in [("weak", weak_manifest), ("gold", gold_manifest), ("test", test_manifest)]:
    for slot in SLOTS:
        n = int(mf[f"{slot}__SeriesInstanceUID"].notna().sum())
        slot_lines.append(f"- {split_name} {slot}: {n}/{len(mf)}")

total_runtime = perf_counter() - RUN_START
fold_df = pd.DataFrame(fold_summary)
macro_gold = gold_ensemble_metrics.iloc[-1]
lines = [
    "RSNA KNEE — BASELINE V1 RESULTS", "=" * 76,
    f"Competition root: {COMP_ROOT}", f"Silver-label path: {SILVER_PATH}",
    f"Series-manifest path: {TRAIN_MANIFEST_PATH}", f"DINO path: {DINO_PATH}",
    f"DINO config: model_type={model_type}, hidden_size={hidden_size}",
    f"GPU(s): {GPU_NAMES}", f"Seed: {SEED}", f"Wall-clock runtime seconds: {total_runtime:.1f}",
    f"Feature-extraction runtime seconds: {FEATURE_RUNTIME:.1f}", "",
    f"Weak train count: {len(weak_manifest)}", f"Gold audit count: {len(gold_manifest)}",
    f"Test count: {len(test_manifest)}", "", "SLOT AVAILABILITY", *slot_lines, "",
    f"Geometry fallback count: {FAIL['geometry_fallback_series']}",
    f"DICOM header failures: {FAIL['dicom_header_failures']}",
    f"DICOM read/decode failures: {FAIL['dicom_decode_failures']}",
    f"Series failures: {FAIL['series_failures']}",
    f"Feature tokens extracted: {FAIL['feature_tokens_extracted']}",
    f"Feature failures: {FAIL['feature_failures']}", "", "FOLDS",
]
for r in fold_summary:
    lines.append(f"- fold {r['fold']}: best_epoch={r['best_epoch']}, weak_val_macro_auc={r['weak_val_macro_auc']:.6f}, gold_macro_auc={r['gold_macro_auc']:.6f}")
lines += ["", "GOLD ENSEMBLE — SMALL-SAMPLE AUDIT ONLY",
          f"Macro AUC: {macro_gold['auc']:.6f}", f"Macro AP: {macro_gold['ap']:.6f}", "Per-label AUC:"]
for _, r in gold_ensemble_metrics.iloc[:-1].iterrows(): lines.append(f"- {r['label']}: {r['auc']:.6f}")
lines += ["", "SUBMISSION", f"Rows: {len(submission)}", f"Columns: {submission.shape[1]}",
          f"Finite check: {bool(np.isfinite(predictions).all())}",
          f"Prediction min/max: {predictions.min():.8f} / {predictions.max():.8f}",
          "Output path: /kaggle/working/submission.csv"]
report = "\n".join(lines)
(OUT / "RESULTS_TO_SEND_BACK.txt").write_text(report, encoding="utf-8")
print(report)

artifact_names = [
    "config.json", "train_feature_manifest.csv", "gold_feature_manifest.csv", "test_feature_manifest.csv",
    "fold_assignments.csv", "weak_cv_metrics.csv", "gold_fold_metrics.csv", "gold_ensemble_metrics.csv",
    "gold_predictions.csv", "submission.csv", "RESULTS_TO_SEND_BACK.txt", "debug_attention_fold0.npz",
] + [f"fold_{i}_head.pt" for i in range(CFG["n_folds"])]
zip_path = Path("/kaggle/working/RSNA_Knee_Baseline_V1_Artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in artifact_names:
        p = OUT / name
        if p.exists(): z.write(p, arcname=name)
print("Artifacts ZIP:", zip_path)
print("Feature caches (not zipped):", weak_cache["path"], gold_cache["path"], test_cache["path"])
